In [1]:
# Nếu thiếu thư viện, chạy trong một cell riêng:
# %pip install xgboost scikit-learn pandas scipy joblib

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from xgboost import XGBClassifier
import joblib

DATA_DIR = Path("/kaggle/input/notebooks/coderntt/data-preprocessing-cctfd/processed_data")
TARGET = "is_fraud"
RANDOM_STATE = 42

In [2]:
train_df = pd.read_csv(DATA_DIR / "train.csv", low_memory=False)
val_df   = pd.read_csv(DATA_DIR / "validation.csv", low_memory=False)
test_df  = pd.read_csv(DATA_DIR / "test.csv", low_memory=False)

for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    if TARGET not in df.columns:
        raise ValueError(f"{name} thiếu cột {TARGET}")
    print(name, df.shape, "| fraud rate:", round(df[TARGET].mean(), 5))

# Target tách riêng; tuyệt đối không để is_fraud nằm trong X
X_train, y_train = train_df.drop(columns=[TARGET]), train_df[TARGET].astype(int)
X_val, y_val = val_df.drop(columns=[TARGET]), val_df[TARGET].astype(int)
X_test, y_test = test_df.drop(columns=[TARGET]), test_df[TARGET].astype(int)

# Đồng bộ schema nếu một split thiếu cột nào đó
feature_cols = list(X_train.columns)
X_val = X_val.reindex(columns=feature_cols)
X_test = X_test.reindex(columns=feature_cols)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in feature_cols if c not in numeric_features]
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

train (1102173, 22) | fraud rate: 0.00578
validation (194502, 22) | fraud rate: 0.00583
test (555719, 22) | fraud rate: 0.00386
Numeric features: 19
Categorical features: 2


In [3]:
# Tương thích cả sklearn mới (sparse_output) và sklearn cũ (sparse)
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# XGBoost dùng scale_pos_weight để xử lý mất cân bằng ban đầu.
# Đây là baseline; có thể thử class_weight/threshold khác sau khi có kết quả.
n_negative = int((y_train == 0).sum())
n_positive = int((y_train == 1).sum())
scale_pos_weight = n_negative / max(n_positive, 1)
print("scale_pos_weight =", round(scale_pos_weight, 2))

print("Đang tiền xử lý X_train và X_val...")
X_train_trans = preprocessor.fit_transform(X_train)
X_val_trans = preprocessor.transform(X_val)

model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    early_stopping_rounds= 100,
    device= "cuda"
)

model.fit(
    X_train_trans,
    y_train,
    eval_set=[(X_val_trans, y_val)],
    verbose=50
)


pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

scale_pos_weight = 171.94
Đang tiền xử lý X_train và X_val...
[0]	validation_0-aucpr:0.62441
[50]	validation_0-aucpr:0.96485
[100]	validation_0-aucpr:0.97744
[150]	validation_0-aucpr:0.98197
[200]	validation_0-aucpr:0.98339
[250]	validation_0-aucpr:0.98450
[300]	validation_0-aucpr:0.98513
[350]	validation_0-aucpr:0.98565
[400]	validation_0-aucpr:0.98589
[450]	validation_0-aucpr:0.98610
[500]	validation_0-aucpr:0.98630
[550]	validation_0-aucpr:0.98655
[600]	validation_0-aucpr:0.98664
[650]	validation_0-aucpr:0.98650
[700]	validation_0-aucpr:0.98665
[701]	validation_0-aucpr:0.98666


In [4]:
def recall_precision_at_top_k(y_true, scores, fraction=0.01):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = max(1, int(np.ceil(len(scores) * fraction)))
    top_idx = np.argsort(scores)[::-1][:k]
    tp = int(y_true[top_idx].sum())
    total_positive = int(y_true.sum())
    precision_at_k = tp / k
    recall_at_k = tp / max(total_positive, 1)
    return {"k": k, "fraud_found": tp,
            "Precision@Top1%": precision_at_k,
            "Recall@Top1%": recall_at_k}

val_scores = pipeline.predict_proba(X_val)[:, 1]
val_pred_05 = (val_scores >= 0.5).astype(int)

print("Validation PR-AUC:", round(average_precision_score(y_val, val_scores), 5))
print("Precision @ threshold 0.5:", round(precision_score(y_val, val_pred_05, zero_division=0), 5))
print("Recall @ threshold 0.5:", round(recall_score(y_val, val_pred_05, zero_division=0), 5))
print("F1 @ threshold 0.5:", round(f1_score(y_val, val_pred_05, zero_division=0), 5))
print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_val, val_pred_05))
print("Top 1% ranking:", recall_precision_at_top_k(y_val, val_scores, 0.01))

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [10:09:11] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Validation PR-AUC: 0.98673
Precision @ threshold 0.5: 0.95065
Recall @ threshold 0.5: 0.96911
F1 @ threshold 0.5: 0.95979
Confusion matrix [ [TN, FP], [FN, TP] ]:
[[193312     57]
 [    35   1098]]
Top 1% ranking: {'k': 1946, 'fraud_found': 1120, 'Precision@Top1%': 0.5755395683453237, 'Recall@Top1%': 0.9885260370697264}


In [5]:
thresholds = np.arange(0.05, 0.96, 0.05)
threshold_results = []
for threshold in thresholds:
    pred = (val_scores >= threshold).astype(int)
    threshold_results.append({
        "threshold": float(threshold),
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0)
    })

threshold_table = pd.DataFrame(threshold_results).sort_values("f1", ascending=False)
display(threshold_table.head(10))
best_threshold = float(threshold_table.iloc[0]["threshold"])
print("Best validation F1 threshold:", best_threshold)

,threshold,precision,recall,f1
13,0.70,0.967111,0.960282,0.963685
12,0.65,0.962930,0.962930,0.962930
14,0.75,0.969561,0.955869,0.962667
17,0.90,0.983410,0.941748,0.962128
15,0.80,0.973779,0.950574,0.962037
11,0.60,0.959578,0.963813,0.961691
10,0.55,0.954704,0.967343,0.960982
16,0.85,0.978042,0.943513,0.960467
9,0.50,0.950649,0.969109,0.959790
8,0.45,0.944254,0.971756,0.957808


Best validation F1 threshold: 0.7000000000000001


In [6]:
# Test chỉ dùng để đánh giá cuối cùng sau khi đã chọn threshold trên validation
test_scores = pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_scores >= best_threshold).astype(int)

print("TEST PR-AUC:", round(average_precision_score(y_test, test_scores), 5))
print("TEST Precision:", round(precision_score(y_test, test_pred, zero_division=0), 5))
print("TEST Recall:", round(recall_score(y_test, test_pred, zero_division=0), 5))
print("TEST F1:", round(f1_score(y_test, test_pred, zero_division=0), 5))
print("TEST confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, test_pred))
print("TEST Top 1% ranking:", recall_precision_at_top_k(y_test, test_scores, 0.01))

TEST PR-AUC: 0.97566
TEST Precision: 0.94757
TEST Recall: 0.92681
TEST F1: 0.93707
TEST confusion matrix [ [TN, FP], [FN, TP] ]:
[[553464    110]
 [   157   1988]]
TEST Top 1% ranking: {'k': 5558, 'fraud_found': 2118, 'Precision@Top1%': 0.3810723281756027, 'Recall@Top1%': 0.9874125874125874}


In [7]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

# 1. Huấn luyện bộ hiệu chỉnh xác suất (Isotonic Calibrator) trên tập Validation
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(val_scores, y_val)

# 2. Tạo xác suất đã hiệu chỉnh trên Validation và Test
val_calib_probs = calibrator.transform(val_scores)
test_calib_probs = calibrator.transform(test_scores)

print("=== KIỂM TRA HIỆU CHỈNH XÁC SUẤT (PROBABILITY CALIBRATION) ===")
print(f"Brier Score (Test) TRƯỚC hiệu chỉnh (Raw Score): {brier_score_loss(y_test, test_scores):.6f}")
print(f"Brier Score (Test) SAU hiệu chỉnh (Calibrated):  {brier_score_loss(y_test, test_calib_probs):.6f}")
print(f"Tỷ lệ gian lận thực tế trên Test:                {y_test.mean():.5f}")
print(f"Trung bình Raw Score trên Test:                  {test_scores.mean():.5f} (Bị phóng đại do scale_pos_weight)")
print(f"Trung bình Calibrated Probability trên Test:     {test_calib_probs.mean():.5f} (Sát thực tế)")

=== KIỂM TRA HIỆU CHỈNH XÁC SUẤT (PROBABILITY CALIBRATION) ===
Brier Score (Test) TRƯỚC hiệu chỉnh (Raw Score): 0.000419
Brier Score (Test) SAU hiệu chỉnh (Calibrated):  0.000377
Tỷ lệ gian lận thực tế trên Test:                0.00386
Trung bình Raw Score trên Test:                  0.00408 (Bị phóng đại do scale_pos_weight)
Trung bình Calibrated Probability trên Test:     0.00380 (Sát thực tế)


In [8]:
# 1. Chọn ngưỡng phân luồng TRÊN TẬP VALIDATION (tuyệt đối không nhìn tập Test)
T_high_raw = float(best_threshold)  # Ngưỡng High Risk (tối ưu F1 trên Validation)
T_low_raw = float(np.percentile(val_scores, 99.0))  # Ngưỡng Medium Risk (vừa khít 1% review capacity)

# Đảm bảo T_low luôn nhỏ hơn hoặc bằng T_high
if T_low_raw >= T_high_raw:
    T_low_raw = float(np.percentile(val_scores, 98.5))

# Ngưỡng tương ứng trên thang xác suất đã hiệu chỉnh (để hiển thị trên Dashboard/Case Brief)
T_high_prob = float(calibrator.transform([T_high_raw])[0])
T_low_prob = float(calibrator.transform([T_low_raw])[0])

print(f"Ngưỡng Raw Score:       T_low = {T_low_raw:.4f} | T_high = {T_high_raw:.4f}")
print(f"Ngưỡng Calibrated Prob: T_low = {T_low_prob:.4f} | T_high = {T_high_prob:.4f}")

# 2. Hàm phân luồng Triage
def assign_triage_tier(scores, t_low, t_high):
    return np.where(
        scores >= t_high, "High Risk",
        np.where(scores >= t_low, "Medium Risk", "Low Risk")
    )

# 3. Đánh giá chất lượng phân luồng (Routing Quality) trên tập TEST
test_tiers = assign_triage_tier(test_scores, T_low_raw, T_high_raw)
tier_summary = pd.DataFrame({
    "y_true": y_test,
    "tier": test_tiers
}).groupby("tier").agg(
    total_cases=("y_true", "count"),
    fraud_cases=("y_true", "sum"),
    legit_cases=("y_true", lambda x: (x == 0).sum())
).reindex(["High Risk", "Medium Risk", "Low Risk"])

tier_summary["tier_rate (%)"] = (tier_summary["total_cases"] / len(y_test) * 100).round(3)
tier_summary["precision_in_tier (%)"] = (tier_summary["fraud_cases"] / tier_summary["total_cases"] * 100).round(2)
tier_summary["share_of_total_fraud (%)"] = (tier_summary["fraud_cases"] / y_test.sum() * 100).round(2)

print("\n=== BÁO CÁO CHẤT LƯỢNG PHÂN LUỒNG (ROUTING QUALITY TRÊN TEST SET) ===")
display(tier_summary)

# 4. Báo cáo gộp cho hàng đợi cần chuyên viên xem xét (Medium + High Risk)
review_mask = test_tiers != "Low Risk"
review_precision = precision_score(y_test, review_mask)
review_recall = recall_score(y_test, review_mask)
review_count = int(review_mask.sum())
review_fp = int(((review_mask == 1) & (y_test == 0)).sum())

print(f"Tổng số case cần review (Medium + High): {review_count} ({review_count/len(y_test)*100:.2f}% tổng giao dịch)")
print(f"Recall của hàng đợi Review (Medium + High):    {review_recall*100:.2f}% ({ int(y_test[review_mask].sum()) }/{ int(y_test.sum()) } vụ gian lận)")
print(f"Precision của hàng đợi Review (Medium + High): {review_precision*100:.2f}% (False Positives: {review_fp} ca)")

Ngưỡng Raw Score:       T_low = 0.0074 | T_high = 0.7000
Ngưỡng Calibrated Prob: T_low = 0.0030 | T_high = 0.5476

=== BÁO CÁO CHẤT LƯỢNG PHÂN LUỒNG (ROUTING QUALITY TRÊN TEST SET) ===


,total_cases,fraud_cases,legit_cases,tier_rate (%),precision_in_tier (%),share_of_total_fraud (%)
tier,,,,,,
High Risk,2098,1988,110,0.378,94.76,92.68
Medium Risk,2095,123,1972,0.377,5.87,5.73
Low Risk,551526,34,551492,99.245,0.01,1.59


Tổng số case cần review (Medium + High): 4193 (0.75% tổng giao dịch)
Recall của hàng đợi Review (Medium + High):    98.41% (2111/2145 vụ gian lận)
Precision của hàng đợi Review (Medium + High): 50.35% (False Positives: 2082 ca)


In [9]:
import xgboost as xgb

# Lấy các thành phần bên trong pipeline đã train
fitted_preprocessor = pipeline.named_steps["preprocessor"]
fitted_xgb_model = pipeline.named_steps["model"]
booster = fitted_xgb_model.get_booster()

# Lấy danh sách tên cột sau khi qua ColumnTransformer (Numeric + OneHot)
transformed_feature_names = fitted_preprocessor.get_feature_names_out()

def explain_single_transaction(df_row, top_k=5):
    """
    Trích xuất Top K đặc trưng đẩy điểm rủi ro gian lận lên cao nhất cho 1 giao dịch.
    Dùng trực tiếp cho Analyst Dashboard và truyền vào Prompt của LLM tạo Case Brief.
    """
    # 1. Transform dòng dữ liệu qua preprocessor
    X_trans = fitted_preprocessor.transform(df_row)
    dmat = xgb.DMatrix(X_trans)
    
    # 2. Tính giá trị đóng góp SHAP (log-odds) bằng native TreeSHAP của XGBoost
    # Cột cuối cùng là bias (base value), các cột trước là đóng góp của từng feature
    contribs = booster.predict(dmat, pred_contribs=True)[0][:-1]
    
    # 3. Gom nhóm lại theo tên feature gốc (gộp các cột one-hot của cùng 1 biến categorical)
    raw_row_dict = df_row.iloc[0].to_dict()
    feature_impacts = {}
    
    for feat_name, shap_val in zip(transformed_feature_names, contribs):
        if feat_name.startswith("num__"):
            orig_col = feat_name.replace("num__", "")
        elif feat_name.startswith("cat__"):
            # Tách tên cột gốc (merchant hoặc category)
            orig_col = "merchant" if feat_name.startswith("cat__merchant_") else "category"
        else:
            orig_col = feat_name
            
        feature_impacts[orig_col] = feature_impacts.get(orig_col, 0.0) + float(shap_val)
    
    # 4. Sắp xếp lấy Top K yếu tố làm TĂNG nguy cơ gian lận mạnh nhất (shap_val > 0)
    sorted_factors = sorted(feature_impacts.items(), key=lambda x: x[1], reverse=True)
    top_risk_factors = []
    for col, impact in sorted_factors[:top_k]:
        if impact > 0:
            top_risk_factors.append({
                "feature": col,
                "actual_value": raw_row_dict.get(col, None),
                "shap_log_odds_impact": round(impact, 4)
            })
            
    return top_risk_factors

# --- CHẠY THỬ TRÊN 1 CA HIGH RISK TRONG TẬP TEST ĐỂ KIỂM TRA ĐẦU RA CHO LLM ---
high_risk_indices = np.where(test_tiers == "High Risk")[0]
sample_idx = high_risk_indices[0]
sample_tx = X_test.iloc[[sample_idx]]

sample_raw_score = float(test_scores[sample_idx])
sample_calib_prob = float(test_calib_probs[sample_idx])
sample_explanation = explain_single_transaction(sample_tx, top_k=5)

print("=== VÍ DỤ OUTPUT CỦA ML MODULE GỬI SANG DASHBOARD & LLM CASE BRIEF ===")
print(f"- Transaction Index:      {sample_idx} (Ground Truth is_fraud = {y_test.iloc[sample_idx]})")
print(f"- Raw Risk Score:         {sample_raw_score:.4f}")
print(f"- Calibrated Probability: {sample_calib_prob:.4f} ({sample_calib_prob*100:.2f}%)")
print(f"- Routing Decision:       {test_tiers[sample_idx]}")
print("- Top Risk Factors (SHAP):")
for item in sample_explanation:
    print(f"   + {item['feature']} = {item['actual_value']} (Đóng góp: +{item['shap_log_odds_impact']})")

=== VÍ DỤ OUTPUT CỦA ML MODULE GỬI SANG DASHBOARD & LLM CASE BRIEF ===
- Transaction Index:      1685 (Ground Truth is_fraud = 1)
- Raw Risk Score:         0.9964
- Calibrated Probability: 0.9189 (91.89%)
- Routing Decision:       High Risk
- Top Risk Factors (SHAP):
   + sum_amt_24h = 1935.19 (Đóng góp: +5.6699)
   + transaction_hour = 22 (Đóng góp: +2.5678)
   + std_amt_7d = 254.7373387838728 (Đóng góp: +1.4945)
   + avg_amt_7d = 160.26724137931035 (Đóng góp: +1.2109)
   + time_since_last_trans_sec = 67219.0 (Đóng góp: +1.0689)


In [10]:
import json

MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_VERSION = "xgb_hist_v2.0"
CONFIG_VERSION = "triage_cfg_v1.0"

# 1. Lưu đầy đủ bundle phục vụ Inference + Calibration + Triage
artifact_bundle = {
    "model_version": MODEL_VERSION,
    "config_version": CONFIG_VERSION,
    "pipeline": pipeline,
    "calibrator": calibrator,
    "thresholds": {
        "t_low_raw": T_low_raw,
        "t_high_raw": T_high_raw,
        "t_low_calibrated": T_low_prob,
        "t_high_calibrated": T_high_prob
    },
    "feature_columns": feature_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features
}

bundle_path = MODEL_DIR / "xgboost_fraud_bundle_v2.joblib"
joblib.dump(artifact_bundle, bundle_path)

# 2. Xuất thêm file config.json độc lập để dễ đọc trên Backend/Git (theo yêu cầu Mục 2.3)
config_metadata = {
    "model_version": MODEL_VERSION,
    "config_version": CONFIG_VERSION,
    "thresholds": artifact_bundle["thresholds"],
    "validation_metrics": {
        "pr_auc": round(float(average_precision_score(y_val, val_scores)), 5),
        "best_f1_threshold": T_high_raw
    },
    "test_metrics": {
        "pr_auc": round(float(average_precision_score(y_test, test_scores)), 5),
        "recall_at_top_1pct": round(float(recall_precision_at_top_k(y_test, test_scores, 0.01)["Recall@Top1%"]), 5),
        "precision_at_top_1pct": round(float(recall_precision_at_top_k(y_test, test_scores, 0.01)["Precision@Top1%"]), 5)
    }
}

config_path = MODEL_DIR / "triage_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_metadata, f, indent=2, ensure_ascii=False)

print("Đã lưu Model Bundle tại:", bundle_path.resolve())
print("Đã lưu Triage Config tại:", config_path.resolve())

Đã lưu Model Bundle tại: /kaggle/working/models/xgboost_fraud_bundle_v2.joblib
Đã lưu Triage Config tại: /kaggle/working/models/triage_config.json


In [12]:
import lightgbm as lgb
from lightgbm import LGBMClassifier

# ==============================================================================
# 1. ĐẢM BẢO DỮ LIỆU ĐÃ ĐƯỢC TRANSFORM QUA PREPROCESSOR
# ==============================================================================
# Tận dụng lại preprocessor đã fit của XGBoost để đồng bộ 100% features
fitted_preprocessor = pipeline.named_steps["preprocessor"]
X_train_trans = fitted_preprocessor.transform(X_train)
X_val_trans   = fitted_preprocessor.transform(X_val)
X_test_trans  = fitted_preprocessor.transform(X_test)

# ==============================================================================
# 2. KHAI BÁO VÀ HUẤN LUYỆN LIGHTGBM VỚI EARLY STOPPING
# ==============================================================================
lgbm_model = LGBMClassifier(
    objective="binary",
    metric="average_precision",   # Tương đương PR-AUC trong LightGBM
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,                # Tương đương độ phức tạp của max_depth=6~8
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=50,         # Chống overfit trên dữ liệu mất cân bằng
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=-1
)

print("Đang huấn luyện LightGBM với Early Stopping...")
lgbm_model.fit(
    X_train_trans,
    y_train,
    eval_set=[(X_val_trans, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

# Đóng gói LightGBM vào Pipeline chuẩn
lgbm_pipeline = Pipeline(steps=[
    ("preprocessor", fitted_preprocessor),
    ("model", lgbm_model)
])

# ==============================================================================
# 3. CHỌN NGƯỠNG F1 TỐI ƯU TRÊN TẬP VALIDATION CHO LIGHTGBM
# ==============================================================================
lgbm_val_scores = lgbm_model.predict_proba(X_val_trans)[:, 1]
lgbm_best_th, lgbm_best_val_f1 = 0.5, 0.0

for th in np.arange(0.05, 0.96, 0.05):
    pred_val = (lgbm_val_scores >= th).astype(int)
    f1_val = f1_score(y_val, pred_val, zero_division=0)
    if f1_val > lgbm_best_val_f1:
        lgbm_best_val_f1 = f1_val
        lgbm_best_th = float(th)

print(f"\nLightGBM - Best Validation F1 Threshold: {lgbm_best_th:.2f} (Val F1 = {lgbm_best_val_f1:.4f})")

# ==============================================================================
# 4. ĐÁNH GIÁ LIGHTGBM TRÊN TẬP TEST
# ==============================================================================
lgbm_test_scores = lgbm_model.predict_proba(X_test_trans)[:, 1]
lgbm_test_pred = (lgbm_test_scores >= lgbm_best_th).astype(int)
lgbm_top1 = recall_precision_at_top_k(y_test, lgbm_test_scores, 0.01)

print("\n=== KẾT QUẢ LIGHTGBM TRÊN TEST SET ===")
print("TEST PR-AUC:", round(average_precision_score(y_test, lgbm_test_scores), 5))
print("TEST Precision:", round(precision_score(y_test, lgbm_test_pred, zero_division=0), 5))
print("TEST Recall:", round(recall_score(y_test, lgbm_test_pred, zero_division=0), 5))
print("TEST F1:", round(f1_score(y_test, lgbm_test_pred, zero_division=0), 5))
print("TEST confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, lgbm_test_pred))
print("TEST Top 1% ranking:", lgbm_top1)

# ==============================================================================
# 5. THỬ NGHIỆM ENSEMBLE (BLENDING 50% XGBOOST + 50% LIGHTGBM)
# ==============================================================================
# Kết hợp điểm dự đoán trên Validation để tìm ngưỡng cho Ensemble
ens_val_scores = 0.5 * val_scores + 0.5 * lgbm_val_scores
ens_best_th, ens_best_val_f1 = 0.5, 0.0
for th in np.arange(0.05, 0.96, 0.05):
    f1_val = f1_score(y_val, (ens_val_scores >= th).astype(int), zero_division=0)
    if f1_val > ens_best_val_f1:
        ens_best_val_f1 = f1_val
        ens_best_th = float(th)

# Đánh giá Ensemble trên Test
ens_test_scores = 0.5 * test_scores + 0.5 * lgbm_test_scores
ens_test_pred = (ens_test_scores >= ens_best_th).astype(int)
ens_top1 = recall_precision_at_top_k(y_test, ens_test_scores, 0.01)

print("\n=== KẾT QUẢ ENSEMBLE (XGBOOST + LIGHTGBM) TRÊN TEST SET ===")
print("ENSEMBLE PR-AUC:", round(average_precision_score(y_test, ens_test_scores), 5))
print("ENSEMBLE Precision:", round(precision_score(y_test, ens_test_pred, zero_division=0), 5))
print("ENSEMBLE Recall:", round(recall_score(y_test, ens_test_pred, zero_division=0), 5))
print("ENSEMBLE F1:", round(f1_score(y_test, ens_test_pred, zero_division=0), 5))
print("ENSEMBLE Top 1% ranking:", ens_top1)

# Lưu lại model LightGBM
joblib.dump(
    {"pipeline": lgbm_pipeline, "threshold": lgbm_best_th, "feature_columns": feature_cols},
    MODEL_DIR / "lightgbm_fraud_model.joblib"
)
print("\nĐã lưu LightGBM tại:", (MODEL_DIR / "lightgbm_fraud_model.joblib").resolve())

Đang huấn luyện LightGBM với Early Stopping...
Training until validation scores don't improve for 100 rounds
[50]	valid_0's average_precision: 0.951804
[100]	valid_0's average_precision: 0.969187
[150]	valid_0's average_precision: 0.974891
[200]	valid_0's average_precision: 0.977338
[250]	valid_0's average_precision: 0.979184
[300]	valid_0's average_precision: 0.979925
[350]	valid_0's average_precision: 0.980206
[400]	valid_0's average_precision: 0.980865
[450]	valid_0's average_precision: 0.981078
[500]	valid_0's average_precision: 0.981369
[550]	valid_0's average_precision: 0.981622
[600]	valid_0's average_precision: 0.981989
[650]	valid_0's average_precision: 0.982083
[700]	valid_0's average_precision: 0.982379
[750]	valid_0's average_precision: 0.982619
[800]	valid_0's average_precision: 0.982896
[850]	valid_0's average_precision: 0.983195
[900]	valid_0's average_precision: 0.983616
[950]	valid_0's average_precision: 0.984055
[1000]	valid_0's average_precision: 0.984316
Did not mee

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



LightGBM - Best Validation F1 Threshold: 0.85 (Val F1 = 0.9605)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== KẾT QUẢ LIGHTGBM TRÊN TEST SET ===
TEST PR-AUC: 0.96996
TEST Precision: 0.96884
TEST Recall: 0.91329
TEST F1: 0.94024
TEST confusion matrix [ [TN, FP], [FN, TP] ]:
[[553511     63]
 [   186   1959]]
TEST Top 1% ranking: {'k': 5558, 'fraud_found': 2104, 'Precision@Top1%': 0.3785534364879453, 'Recall@Top1%': 0.9808857808857809}

=== KẾT QUẢ ENSEMBLE (XGBOOST + LIGHTGBM) TRÊN TEST SET ===
ENSEMBLE PR-AUC: 0.97612
ENSEMBLE Precision: 0.95839
ENSEMBLE Recall: 0.92354
ENSEMBLE F1: 0.94065
ENSEMBLE Top 1% ranking: {'k': 5558, 'fraud_found': 2117, 'Precision@Top1%': 0.38089240734077007, 'Recall@Top1%': 0.9869463869463869}

Đã lưu LightGBM tại: /kaggle/working/models/lightgbm_fraud_model.joblib


In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from lightgbm import LGBMClassifier

# ==============================================================================
# 1. XÂY DỰNG BASELINE DỰA TRÊN LUẬT ĐƠN GIẢN (SIMPLE RULE-BASED HEURISTIC)
# ==============================================================================
def rule_based_scoring(df):
    """
    Hệ thống luật nghiệp vụ (Heuristic Rules) chấm điểm rủi ro từ 0.0 đến 1.0:
    - Luật 1 (+0.40): Số tiền giao dịch lớn (> $500)
    - Luật 2 (+0.25): Giao dịch diễn ra vào khung giờ nhạy cảm (0h - 4h sáng)
    - Luật 3 (+0.20): Số tiền đột biến gấp > 5 lần mức trung bình 7 ngày qua
    - Luật 4 (+0.15): Tần suất quẹt thẻ dồn dập (>= 3 lần trong 1 giờ qua)
    """
    score = np.zeros(len(df), dtype=float)
    
    if "amt" in df.columns:
        score += np.where(df["amt"] > 500, 0.40, np.where(df["amt"] > 250, 0.20, 0.0))
    if "transaction_hour" in df.columns:
        score += np.where(df["transaction_hour"].isin([0, 1, 2, 3, 4]), 0.25, 0.0)
    if "amt_to_avg_7d_ratio" in df.columns:
        score += np.where(df["amt_to_avg_7d_ratio"] > 5.0, 0.20, 0.0)
    if "trans_count_1h" in df.columns:
        score += np.where(df["trans_count_1h"] >= 3, 0.15, 0.0)
        
    # Tie-break nhẹ theo số tiền giao dịch khi xếp hạng Top 1%
    if "amt" in df.columns:
        score += np.clip(df["amt"] / 100000.0, 0, 0.04)
        
    return np.clip(score, 0.0, 1.0)

rule_test_scores = rule_based_scoring(X_test)
rule_test_pred = (rule_test_scores >= 0.50).astype(int)


# ==============================================================================
# 2. XÂY DỰNG BASELINE ML TRUYỀN THỐNG: LOGISTIC REGRESSION
# ==============================================================================
if "lr_test_scores" not in globals():
    lr_numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    lr_preprocessor = ColumnTransformer(
        transformers=[
            ("num", lr_numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop"
    )

    lr_pipeline = Pipeline(steps=[
        ("preprocessor", lr_preprocessor),
        ("model", LogisticRegression(
            class_weight="balanced",
            solver="saga",
            max_iter=200,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ])

    print("Đang huấn luyện Baseline Logistic Regression...")
    lr_pipeline.fit(X_train, y_train)

    lr_val_scores = lr_pipeline.predict_proba(X_val)[:, 1]
    lr_best_th, lr_best_val_f1 = 0.5, 0.0
    for th in np.arange(0.05, 0.96, 0.05):
        f1_val = f1_score(y_val, (lr_val_scores >= th).astype(int), zero_division=0)
        if f1_val > lr_best_val_f1:
            lr_best_val_f1 = f1_val
            lr_best_th = float(th)

    lr_test_scores = lr_pipeline.predict_proba(X_test)[:, 1]
    lr_test_pred = (lr_test_scores >= lr_best_th).astype(int)

    joblib.dump(
        {"pipeline": lr_pipeline, "threshold": lr_best_th, "feature_columns": feature_cols},
        MODEL_DIR / "logistic_regression_baseline.joblib"
    )
else:
    print("Đã lấy kết quả Logistic Regression có sẵn trong bộ nhớ.")


# ==============================================================================
# 3. LIGHTGBM & ENSEMBLE (TỰ ĐỘNG DÙNG LẠI NẾU ĐÃ CHẠY Ở CELL TRƯỚC)
# ==============================================================================
if "lgbm_test_scores" not in globals() or "ens_test_scores" not in globals():
    fitted_preprocessor = pipeline.named_steps["preprocessor"]
    X_train_trans = fitted_preprocessor.transform(X_train)
    X_val_trans   = fitted_preprocessor.transform(X_val)
    X_test_trans  = fitted_preprocessor.transform(X_test)

    lgbm_model = LGBMClassifier(
        objective="binary",
        metric="average_precision",
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=50,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=-1
    )

    print("Đang huấn luyện LightGBM...")
    lgbm_model.fit(
        X_train_trans, y_train,
        eval_set=[(X_val_trans, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    lgbm_val_scores = lgbm_model.predict_proba(X_val_trans)[:, 1]
    lgbm_best_th, lgbm_best_val_f1 = 0.5, 0.0
    for th in np.arange(0.05, 0.96, 0.05):
        f1_val = f1_score(y_val, (lgbm_val_scores >= th).astype(int), zero_division=0)
        if f1_val > lgbm_best_val_f1:
            lgbm_best_val_f1 = f1_val
            lgbm_best_th = float(th)

    lgbm_test_scores = lgbm_model.predict_proba(X_test_trans)[:, 1]
    lgbm_test_pred = (lgbm_test_scores >= lgbm_best_th).astype(int)

    # Tính Ensemble (50% XGBoost + 50% LightGBM)
    ens_val_scores = 0.5 * val_scores + 0.5 * lgbm_val_scores
    ens_best_th, ens_best_val_f1 = 0.5, 0.0
    for th in np.arange(0.05, 0.96, 0.05):
        f1_val = f1_score(y_val, (ens_val_scores >= th).astype(int), zero_division=0)
        if f1_val > ens_best_val_f1:
            ens_best_val_f1 = f1_val
            ens_best_th = float(th)

    ens_test_scores = 0.5 * test_scores + 0.5 * lgbm_test_scores
    ens_test_pred = (ens_test_scores >= ens_best_th).astype(int)
else:
    print("Đã lấy kết quả LightGBM và Ensemble có sẵn trong bộ nhớ.")


# ==============================================================================
# 4. HÀM TỔNG HỢP CHỈ SỐ & LẬP BẢNG ĐỐI CHIẾU 5 MÔ HÌNH TRÊN TEST SET
# ==============================================================================
def evaluate_for_table(model_name, y_true, scores, preds, threshold_used):
    top1 = recall_precision_at_top_k(y_true, scores, 0.01)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return {
        "Model": model_name,
        "Threshold": round(float(threshold_used), 2),
        "PR-AUC": round(average_precision_score(y_true, scores), 5),
        "Precision": round(precision_score(y_true, preds, zero_division=0), 5),
        "Recall": round(recall_score(y_true, preds, zero_division=0), 5),
        "F1-Score": round(f1_score(y_true, preds, zero_division=0), 5),
        "Recall@Top1%": round(top1["Recall@Top1%"], 5),
        "Precision@Top1%": round(top1["Precision@Top1%"], 5),
        "Fraud@Top1%": f"{top1['fraud_found']}/{int(y_true.sum())}",
        "TP / FP / FN": f"{tp} / {fp} / {fn}"
    }

comparison_df = pd.DataFrame([
    evaluate_for_table("1. Simple Rule-Based (Heuristic)", y_test, rule_test_scores, rule_test_pred, 0.50),
    evaluate_for_table("2. Logistic Regression (Baseline ML)", y_test, lr_test_scores, lr_test_pred, lr_best_th),
    evaluate_for_table("3. LightGBM + Historical Features", y_test, lgbm_test_scores, lgbm_test_pred, lgbm_best_th),
    evaluate_for_table("4. XGBoost + Historical Features", y_test, test_scores, test_pred, best_threshold),
    evaluate_for_table("5. Ensemble (XGBoost + LightGBM)", y_test, ens_test_scores, ens_test_pred, ens_best_th),
])

print("\n=== BẢNG ĐỐI CHIẾU CHẤT LƯỢNG 5 MÔ HÌNH TRÊN TEST SET (MỤC 2.1 & 4.2) ===")
display(comparison_df)

# Xuất bảng ra CSV để tiện copy vào Word/Slide báo cáo
report_csv_path = MODEL_DIR / "model_comparison_test_results.csv"
comparison_df.to_csv(report_csv_path, index=False)
print("Đã xuất bảng kết quả ra file CSV tại:", report_csv_path.resolve())

Đã lấy kết quả Logistic Regression có sẵn trong bộ nhớ.
Đã lấy kết quả LightGBM và Ensemble có sẵn trong bộ nhớ.

=== BẢNG ĐỐI CHIẾU CHẤT LƯỢNG 5 MÔ HÌNH TRÊN TEST SET (MỤC 2.1 & 4.2) ===


,Model,Threshold,PR-AUC,Precision,Recall,F1-Score,Recall@Top1%,Precision@Top1%,Fraud@Top1%,TP / FP / FN
0,1. Simple Rule-Based (Heuristic),0.50,0.10644,0.11134,0.32075,0.16531,0.28998,0.11191,622/2145,688 / 5491 / 1457
1,2. Logistic Regression (Baseline ML),0.95,0.32148,0.32485,0.71422,0.44658,0.76457,0.29507,1640/2145,1532 / 3184 / 613
2,3. LightGBM + Historical Features,0.85,0.96996,0.96884,0.91329,0.94024,0.98089,0.37855,2104/2145,1959 / 63 / 186
3,4. XGBoost + Historical Features,0.70,0.97566,0.94757,0.92681,0.93707,0.98741,0.38107,2118/2145,1988 / 110 / 157
4,5. Ensemble (XGBoost + LightGBM),0.70,0.97612,0.95839,0.92354,0.94065,0.98695,0.38089,2117/2145,1981 / 86 / 164


Đã xuất bảng kết quả ra file CSV tại: /kaggle/working/models/model_comparison_test_results.csv
